<a href="https://colab.research.google.com/github/abdulrahman0700/intern_FlyRank_ai/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulrahman0700/intern_FlyRank_ai/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Claim:** one row = one client page from `data/raw/content_refresh_anonymized.csv` (the anonymized starter slice, ~30,000 rows). Each row is a rolled-up performance/trend snapshot for that page as of the export date — it is **not** one row per day; the daily granularity only exists in the full warehouse release (`fact_content_daily_performance`, used from Week 3 onward via DuckDB/Hugging Face).

**Time window:** the snapshot aggregates each page's GSC/GA4 history up to the export date. Per `GUIDE.md`, per-client history depth is **unbalanced** — some clients have longer GSC/GA4 history than others (see `dim_clients.gsc_data_start` / `ga4_data_start` in the full release; the starter CSV likely encodes an equivalent per-row date field — confirmed below). Every claim here gets checked against the actual file in Section 3, not assumed.

In [ ]:
import pandas as pd

DATA_PATH = "/content/content_refresh_anonymized.csv"  # run this notebook from the repo root
df = pd.read_csv(DATA_PATH)

print("shape:", df.shape)
print("\ncolumns:")
print(list(df.columns))

# grain check: does one row really = one page?
id_candidates = [c for c in df.columns if "id" in c.lower() or "page" in c.lower() or "url" in c.lower()]
print("\ncandidate identifier columns:", id_candidates)
for c in id_candidates:
    print(f"  {c}: {df[c].nunique()} unique / {len(df)} rows")

# date/window columns, if any
date_candidates = [c for c in df.columns if "date" in c.lower() or "start" in c.lower() or "end" in c.lower()]
print("\ncandidate date columns:", date_candidates)
for c in date_candidates:
    print(f"  {c}: min={df[c].min()}  max={df[c].max()}")

shape: (30000, 44)

columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

candidate identifier columns: ['content_id', 'client_id', 'provider_used', 'pageviews_90d']
  content_id: 30000 unique / 30000 rows
  client_id: 32 unique / 30000 rows
  provider_used: 2 unique / 300

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Label:** `is_declining_label` — `(trend_direction == "down")`. This is the observed outcome we're trying to predict.
- **Excluded (leakage):** `trend_direction`, `trend_pct` — these are the columns the label is *derived from*. Including either as a feature would let the model see the answer. (`GUIDE.md` calls this out explicitly and Notebook 02 demonstrates the failure mode.)
- **Context (not features, used for grouping/identification only):** client/page identifier columns, and any anonymized URL or domain hash — needed for the client-holdout split (train/test splits by *client*, not by row) but never fed to the model.
- **Features:** the remaining performance and content signals (GSC metrics like clicks/impressions/CTR/position, GA4 engagement metrics, content-freshness signals like word count or last-updated date, etc.) — exact list confirmed against the real columns below, cross-checked with `docs/data-dictionary.md` so nothing gets miscategorized.

Fill the table below from the live column list once Section 1's code cell has run.

In [ ]:
label_col = "is_declining_label"
# Add the label column to the DataFrame first
df[label_col] = (df["trend_direction"] == "down")

excluded_cols = ["trend_direction", "trend_pct"]  # define the label -> leakage if used as features
context_cols = id_candidates  # identifiers used for grouping/holdout, not modeling

feature_cols = [
    c for c in df.columns
    if c not in [label_col] + excluded_cols + context_cols
]

print(f"label ({1}): {label_col}")
print(f"excluded ({len(excluded_cols)}): {excluded_cols}")
print(f"context ({len(context_cols)}): {context_cols}")
print(f"feature candidates ({len(feature_cols)}): {feature_cols}")

assert len(feature_cols) + len(excluded_cols) + len(context_cols) + 1 == df.shape[1], \
    "bucket counts don't add up to total columns — re-check the split"

label (1): is_declining_label
excluded (2): ['trend_direction', 'trend_pct']
context (4): ['content_id', 'client_id', 'provider_used', 'pageviews_90d']
feature candidates (38): ['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'model_used', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*



In [ ]:
# --- Claim: one row = one page (grain) ---
print("row count:", len(df))
for c in id_candidates:
    dupes = df[c].duplicated().sum()
    print(f"{c}: {dupes} duplicate values ({'PASS - unique grain' if dupes == 0 else 'FAIL - not one row per page'})")

# --- Claim: label is derived correctly ---
recomputed = (df["trend_direction"] == "down")
matches = (recomputed == df[label_col]).mean()
print(f"\nlabel formula match rate: {matches:.4%}")

# --- Missing values across all columns ---
missing = df.isna().mean().sort_values(ascending=False)
print("\ntop columns by % missing:")
print(missing.head(10))

# --- Class balance on the label ---
print("\nlabel distribution:")
print(df[label_col].value_counts(normalize=True))

# --- Window/date spread, if a date column exists ---
for c in date_candidates:
    print(f"\n{c} spread:")
    print(pd.to_datetime(df[c], errors="coerce").describe())

row count: 30000
content_id: 0 duplicate values (PASS - unique grain)
client_id: 29968 duplicate values (FAIL - not one row per page)
provider_used: 29997 duplicate values (FAIL - not one row per page)
pageviews_90d: 29144 duplicate values (FAIL - not one row per page)

label formula match rate: 100.0000%

top columns by % missing:
provider_used        0.714600
word_count           0.256633
char_count           0.256633
char_count_tier      0.256633
word_count_tier      0.256633
model_used           0.191100
trend_pct            0.112933
competition_level    0.087000
cpc                  0.082267
search_volume        0.082267
dtype: float64

label distribution:
is_declining_label
True     0.542067
False    0.457933
Name: proportion, dtype: float64

days_since_last_update spread:
count                            30000
mean     1970-01-01 00:00:00.000000046
min      1970-01-01 00:00:00.000000001
25%      1970-01-01 00:00:00.000000020
50%      1970-01-01 00:00:00.000000020
75%      1970-0

/tmp/ipykernel_1517/1542377008.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  print(pd.to_datetime(df[c], errors="coerce").describe())


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Anonymized, aggregated slice only.** No client names, domains, URLs, or raw queries — by design (`DATA_USE.md`). Any pattern found here is *observed/directional*, never a claim about a specific real client.
- **Sample vs. full history.** This is ~30k rows out of the full warehouse's ~79M-row daily fact table. Rare events, seasonal effects, and long-tail client behavior may simply not be represented here.
- **Unbalanced panel.** Per `GUIDE.md`, client history depth differs — some clients have more GSC/GA4 history than others (`gsc_data_start` / `ga4_data_start` in the full release). Comparisons across pages may implicitly compare pages with very different amounts of history behind them; confirm whether the starter CSV surfaces this per row (Section 1/3), and if it doesn't, treat it as an unverifiable blind spot rather than assuming balance.
- **Single-snapshot aggregation, not a time series.** This file is a rolled-up snapshot per page, not day-by-day. It can support "is this page currently declining" scoring, but it cannot support causal statements about *when* or *why* a decline started — that needs the daily fact table.
- **Label is a proxy, not ground truth on intent.** `is_declining_label` reflects a measured trend, not confirmed causes (algorithm change, content decay, competitor movement, seasonality) — framing must stay in the observed/measured/directional/decision-support register, never "predicted the algorithm."
- **No cross-checked data dictionary in this pass.** I didn't have access to `docs/data-dictionary.md` or the exact `writing-data-contracts` / `flyrank-data` skill text when drafting this — the column-level claims above should be read as a starting scaffold to verify against those two files, not a final source of truth.

In [ ]:
for c in date_candidates:
    print(f"{c} spread (days from min to max):")
    dt = pd.to_datetime(df[c], errors="coerce")
    print((dt.max() - dt.min()))

# Sample size vs. full warehouse (~79M daily rows) — just to keep the scale claim honest
print(f"\nstarter rows: {len(df):,}")
print("full warehouse (fact_content_daily_performance): ~78,835,655 rows (per FlyRank/internship-warehouse)")

days_since_last_update spread (days from min to max):
0 days 00:00:00.000000372
trend_direction spread (days from min to max):
NaT
trend_pct spread (days from min to max):
0 days 00:00:00.000045

starter rows: 30,000
full warehouse (fact_content_daily_performance): ~78,835,655 rows (per FlyRank/internship-warehouse)


/tmp/ipykernel_1517/681429130.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dt = pd.to_datetime(df[c], errors="coerce")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.